In [ ]:
import os, sys
from pyspark.sql import functions as SF
import duckdb


In [3]:
# Cria a conexão Spark

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

In [10]:
# Cria a conexão com o DuckDB

con = duckdb.connect(database=':memory:')

con.execute("SET max_memory = '10GB';")
con.execute("SET threads = 4;")

In [4]:
PROJECT_PATH    = os.getcwd()
ROOT_DATA_PATH  = "C:\\Marco Conti\\Projetos\\Dados\\"

In [6]:
def write_data(df_, write_path, prefix_file_name):
    # df_temperatura_final.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_t2m_temperatura.csv", index=False)
    df_.toPandas().to_parquet(f"{write_path}\\{prefix_file_name}.parquet")

# Somente para processamento LOCAL, pode ser descartada
def write_data_csv(df_write, write_path, file_name):
    df_write.toPandas().to_csv(f"{write_path}\{file_name}", index=False)

Seleciona a grade de municípios


In [7]:
df_grid_municipio = spark.read.parquet(r"C:\Marco Conti\Projetos\mais_einstein\Municipio\d_lookup_grid_municipio_brasil.parquet")
df_grid_municipio.printSchema()
# df_grid_municipio.show(10, False)

root
 |-- id_geo: string (nullable = true)
 |-- latitude_centro: double (nullable = true)
 |-- longitude_centro: double (nullable = true)
 |-- code_muni: double (nullable = true)
 |-- name_muni: string (nullable = true)
 |-- abbrev_state: string (nullable = true)
 |-- area_municipio_m2: double (nullable = true)
 |-- area_interseccao_m2: double (nullable = true)
 |-- fator_peso_municipio: double (nullable = true)



Faz a junção dos municípios e temperatura aplicando o peso ponderado para o município

In [ ]:
path_temperatura    = "C://Marco Conti//Projetos//Dados//ERA5-temperaturas//arquivos_parquet/*.parquet"
path_municipios     = "C://Marco Conti//Projetos//mais_einstein//Municipio//d_lookup_grid_municipio_brasil.parquet"
path_write_parquet  = "C://Marco Conti//Projetos//Dados//temperaturas_por_municipios_SEM_georrefenciamento.parquet"

duckdb_sql = \
   f"""COPY (
            SELECT 
                l.code_muni,
                l.name_muni,
                l.abbrev_state AS uf,
                f.data_medicao::DATE AS data_medicao,
                YEAR(f.data_medicao::DATE) AS ANO,
                ROUND(SUM(f.valor * l.fator_peso_municipio), 2) AS temp_media_municipio
            FROM read_parquet('{path_temperatura}') f
            INNER JOIN read_parquet('{path_municipios}') l
            ON ROUND(f.latitude, 2) = ROUND(l.latitude_centro, 2)
            AND ROUND(f.longitude, 2) = ROUND(l.longitude_centro, 2)
            GROUP BY 
                l.code_muni, 
                l.name_muni, 
                l.abbrev_state, 
                f.data_medicao::DATE
        ) TO '{path_write_parquet}' 
        (FORMAT PARQUET, OVERWRITE_OR_IGNORE);
    """
# (FORMAT CSV, HEADER TRUE, PARTITION_BY (ANO), OVERWRITE_OR_IGNORE);

print("Executando transformação e gravação em PARQUET via DuckDB...")
con.execute(duckdb_sql)
print("Concluído!")


Executando transformação e gravação em CSV via DuckDB...
Concluído!


In [8]:
df_p = spark.read.parquet(r"C:\Marco Conti\Projetos\Dados\temperaturas_por_municipios_Sem_georrefenciamento.parquet")
df_p.printSchema()
print(df_p.count()) # 63.676.240


root
 |-- code_muni: double (nullable = true)
 |-- name_muni: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- data_medicao: date (nullable = true)
 |-- ANO: long (nullable = true)
 |-- temp_media_municipio: double (nullable = true)

63676240


In [ ]:
path_temperatura    = r"C:\Marco Conti\Projetos\Dados\temperaturas_por_municipios_Sem_georrefenciamento.parquet"
path_munic_georefer = r"C:\Marco Conti\Projetos\mais_einstein\Municipio\d_lookup_grid_municipio_brasil.parquet"
path_write          = r"C:\Marco Conti\Projetos\Dados\temperaturas_por_municipios_COM_georrefenciamento.parquet" 

duckdb_sql_temp_munic_georref = \
    f"""COPY (
            SELECT 
                l.code_muni,
                l.name_muni,
                l.id_geo,
                l.latitude_centro,
                l.longitude_centro,
                t.uf,
                t.data_medicao,
                t.ANO,
                t.temp_media_municipio
             FROM read_parquet('{path_temperatura}') t
            INNER JOIN read_parquet('{path_munic_georefer}') l
               ON l.code_muni = t.code_muni
        )
        TO 'C://Marco Conti//Projetos//Dados//temperaturas_por_municipios_COM_georrefenciamento.parquet' 
        (FORMAT PARQUET, OVERWRITE_OR_IGNORE);
     """

# (FORMAT CSV, HEADER TRUE, PARTITION_BY (ANO), OVERWRITE_OR_IGNORE);

print("Junção entre Temperatura e dados de georreferenciamento por municipio")
con.execute(duckdb_sql_temp_munic_georref)
print("Concluído!")


Junção entre Temperatura e dados de georreferenciamento por municipio
Concluído!


In [15]:
path_ = r"C:\Marco Conti\Projetos\Dados\temperaturas_por_municipios_COM_georrefenciamento.parquet"
df_p  = spark.read.parquet(path_)
df_p.printSchema()
df_p.count() # 391.477.408

root
 |-- code_muni: double (nullable = true)
 |-- name_muni: string (nullable = true)
 |-- id_geo: string (nullable = true)
 |-- latitude_centro: double (nullable = true)
 |-- longitude_centro: double (nullable = true)
 |-- uf: string (nullable = true)
 |-- data_medicao: date (nullable = true)
 |-- ANO: long (nullable = true)
 |-- temp_media_municipio: double (nullable = true)



391477408

In [17]:
# df_p.select("code_muni", "ano").groupBy("code_muni").agg(SF.countDistinct("ano")).show(100,False)

df_p.filter("name_muni = 'São Paulo' and data_medicao = '2025-07-01'").show(10,False)

+---------+---------+------------------+---------------+----------------+---+------------+----+--------------------+
|code_muni|name_muni|id_geo            |latitude_centro|longitude_centro|uf |data_medicao|ANO |temp_media_municipio|
+---------+---------+------------------+---------------+----------------+---+------------+----+--------------------+
|3550308.0|São Paulo|GRID_-23.25_-46.50|-23.25         |-46.5           |SP |2025-07-01  |2025|14.47               |
|3550308.0|São Paulo|GRID_-23.50_-46.25|-23.5          |-46.25          |SP |2025-07-01  |2025|14.47               |
|3550308.0|São Paulo|GRID_-23.50_-46.50|-23.5          |-46.5           |SP |2025-07-01  |2025|14.47               |
|3550308.0|São Paulo|GRID_-23.50_-46.75|-23.5          |-46.75          |SP |2025-07-01  |2025|14.47               |
|3550308.0|São Paulo|GRID_-23.75_-46.50|-23.75         |-46.5           |SP |2025-07-01  |2025|14.47               |
|3550308.0|São Paulo|GRID_-23.75_-46.75|-23.75         |-46.75  

In [18]:
df_20250701 = df_p.filter("data_medicao = '2025-07-01'")
df_20250701.count()

34244

In [ ]:
# Para converter o  DataFrame do PySpark para um arquivo ou objeto GeoJSON

import geopandas as gpd
from shapely.geometry import Point

# 1. Converter o Spark DataFrame para Pandas DataFrame
pandas_df = df_20250701.toPandas()

# 2. Criar a coluna de geometria a partir das coordenadas
geometry = [Point(xy) for xy in zip(pandas_df['longitude_centro'], pandas_df['latitude_centro'])]

# 3. Criar o GeoDataFrame especificando o sistema de coordenadas WGS84 (EPSG:4326)
gdf = gpd.GeoDataFrame(pandas_df, geometry=geometry, crs="EPSG:4326")

# Opcional: Converter a data para string para evitar erros de serialização no JSON
gdf['data_medicao'] = gdf['data_medicao'].astype(str)

# 4a. Salvar diretamente como arquivo .geojson
gdf.to_file("dados_municipio.geojson", driver="GeoJSON")

# # 4b. Ou obter o resultado como uma string/dicionário GeoJSON em memória
# geojson_str = gdf.to_json()

In [ ]:
df_grid_municipio.createOrReplaceTempView("d_lookup_grid_municipio")
df_temperatura.createOrReplaceTempView("temp_temperatura")


# query_temp_SP = \
#     """ SELECT l.code_muni,
#             l.name_muni,
#             l.abbrev_state AS uf,
#             to_date(f.data_medicao,'yyyy-MM-dd') as data_medicao,
#             YEAR(to_date(f.data_medicao,'yyyy-MM-dd')) as ANO,
#             --year(f.data_medicao) as ANO,
#             -- Aplica o peso para cada coordenada (lat. e long.) contidas no Shape do Município
#             -- e então faz soma todas os resultados
#             ROUND(SUM(f.valor * l.fator_peso_municipio), 2) AS temp_media_municipio
#           FROM d_lookup_grid_municipio l 
#           left JOIN temp_temperatura f
#             -- Arredondamento Garante o encaixe perfeito das coordenadas numéricas
#             ON ROUND(f.latitude,  2) = ROUND(l.latitude_centro, 2)
#            AND ROUND(f.longitude, 2) = ROUND(l.longitude_centro,2)
#          where 1=1
#            -- and f.data_medicao = '2025-07-01'  
#            -- and l.name_muni = 'São Paulo'
#          GROUP BY 
#             l.code_muni, 
#             l.name_muni, 
#             l.abbrev_state, 
#             f.data_medicao
#     """

query_temp_SP = \
    """SELECT /*+ BROADCAST(l) */
            l.code_muni,
            l.name_muni,
            l.abbrev_state AS uf,
            f.dt_medicao AS data_medicao,
            YEAR(f.dt_medicao) AS ANO,
            ROUND(SUM(f.valor * l.fator_peso_municipio), 2) AS temp_media_municipio
        FROM (
            SELECT 
                ROUND(latitude, 2) AS lat_rnd,
                ROUND(longitude, 2) AS lon_rnd,
                valor,
                TO_DATE(data_medicao, 'yyyy-MM-dd') AS dt_medicao
            FROM temp_temperatura
        ) f
        INNER JOIN (
            SELECT 
                code_muni,
                name_muni,
                abbrev_state,
                fator_peso_municipio,
                ROUND(latitude_centro, 2) AS lat_rnd,
                ROUND(longitude_centro, 2) AS lon_rnd
            FROM d_lookup_grid_municipio
        ) l 
          ON f.lat_rnd = l.lat_rnd
        AND f.lon_rnd = l.lon_rnd
        GROUP BY 
            l.code_muni, 
            l.name_muni, 
            l.abbrev_state, 
            f.dt_medicao
    """

df_temp = spark.sql(query_temp_SP)
df_temp.printSchema()

df_temp.filter("ano = 2000").show(10,False)


In [ ]:
write_path = r"C:\Marco Conti\Projetos\Dados\temperaturas_por_municipios"
df_temp.write \
    .mode("overwrite") \
    .partitionBy("ANO") \
    .parquet(write_path)

In [ ]:

# # "C:\Marco Conti\Projetos\Dados\ERA5-temperaturas"
# write_path = f"{ROOT_DATA_PATH}ERA5-temperaturas"

# for ano in range(1995,2027):
#     df_temp_write = df_temp.filter(f"ano = {ano}")
#     prefix_file_name = f"temperatura_por_municipio_{ano}.parquet"

#     write_data(df_temp, write_path, prefix_file_name)

In [ ]:
# import pyspark
# print("Versão do Spark:", pyspark.__version__)

# # Descobre a versão compilada do Hadoop
# spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()